# Phase 2(보강) — 라오스 공식 통계 수집 (ASEANstats / LSB)

라오스 현지 정량 근거 보강. **ASEANstats**(LSB 차량 데이터 미러)의 REST API로 등록 차량·이륜차를 수집하고,
Customs EV 수입(4,437대)과 결합해 **EV 침투율·이륜차 비중**을 계산합니다.

> 신뢰도: ASEANstats = LSB 출처(공식). EV 4,437대 = Customs(2024.10, Laotian Times 인용).
> 수집 계획: `../docs/lsb_data_acquisition_plan.md` · 분석: `../outputs/laos_official_stats_analysis.md`


In [ ]:
# !pip install requests pandas

In [ ]:
import requests, pandas as pd

BASE = 'https://data.aseanstats.org/api/indicator'
IND = {'vehicles_total': 'ASE.TRP.ROD.B.005', 'motorcycles': 'ASE.TRP.ROD.B.011'}
LAO_KEYS = ('Lao', 'LAO', 'Laos')   # 국가명 표기 변형 대응

def fetch_laos(indicator_code):
    r = requests.get(f'{BASE}/{indicator_code}', timeout=30)
    r.raise_for_status()
    data = r.json()
    rows = data.get('data', data if isinstance(data, list) else [])
    out = {}
    for rec in rows:
        name = str(rec.get('country') or rec.get('ref_area') or rec.get('name') or '')
        if any(k in name for k in LAO_KEYS):
            yr = rec.get('year') or rec.get('time') or rec.get('TIME_PERIOD')
            val = rec.get('value') or rec.get('obs_value') or rec.get('OBS_VALUE')
            if yr and val is not None:
                out[int(yr)] = float(val)
    return out

# ⚠️ ASEANstats 응답 스키마가 바뀔 수 있어, 실패 시 아래 '검증된 수집값'으로 폴백
try:
    veh = fetch_laos(IND['vehicles_total'])
    moto = fetch_laos(IND['motorcycles'])
    assert veh and moto
    print('API 수집 성공')
except Exception as e:
    print('API 수집 실패 → 폴백 사용:', e)
    veh  = {2021:2544.894, 2022:3116.546, 2023:3160.534, 2024:3160.0}   # 천 대
    moto = {2021:1841.618, 2022:2299.116, 2023:2369.514, 2024:2448.254} # 천 대

## 차량 통계 정리 + 이륜차 비중


In [ ]:
df = pd.DataFrame({'총등록차량_천대': veh, '이륜차_천대': moto}).sort_index()
df['이륜차비중'] = (df['이륜차_천대'] / df['총등록차량_천대'] * 100).round(1)
df.tail(5)

## EV 침투율 (Customs EV 수입 ÷ LSB 총 등록차량)


In [ ]:
EV_IMPORTS_2024 = 4437          # 라오스 Customs 누적(2024.10), Laotian Times 인용
total_2024 = df.loc[2024, '총등록차량_천대'] * 1000
moto_2024  = df.loc[2024, '이륜차_천대'] * 1000
ev_pen = EV_IMPORTS_2024 / total_2024 * 100
print(f'2024 총 등록차량 : {total_2024:,.0f} 대')
print(f'2024 이륜차      : {moto_2024:,.0f} 대 ({df.loc[2024,"이륜차비중"]}%)')
print(f'EV 수입(누적)    : {EV_IMPORTS_2024:,} 대')
print(f'→ EV 침투율      : {ev_pen:.3f}%  (극초기 시장)')

## 인사이트
- **EV 침투율 0.14%** → 극초기, 선점 효과 큼.
- **이륜차 77.5%(244.8만 대)** + 충전소 부재 → **이륜차 EV 충전 미개척 시장**.
- TAM = ‘EV 4,437대’가 아니라 **316만 모빌리티 시장의 EV 전환**으로 프레임.


In [ ]:
df.to_csv('../outputs/laos_vehicle_stats.csv', encoding='utf-8-sig')
print('저장: outputs/laos_vehicle_stats.csv')